# Harvard's Endowment: Mean-Variance Allocation and TIPS

**FINM 36700 Portfolio and Risk Management Week 1**
**Materials & Sources**
- Course case-study page: [Harvard's Endowment](https://markhendricks.github.io/finm-portfolio/case_studies/Harvard%27s%20Endowment.html)
- HBS case: *The Harvard Management Company and Inflation-Protected Bonds*


# 1. HMC's Approach

## 1.1 Why not optimize directly across 1,000 securities?

A full MV optimization over 1,000 securities is mainly difficult because of estimation for the number of risk parameters. With \(N=1000\) securities, HMC would need roughly $\frac{N(N+1)}{2}=500500$ unique covariance-matrix entries, in addition to 1,000 expected returns. Those inputs have to be estimated from a finite return history. As the dimension grows, the covariance matrix becomes increasingly noisy and potentially ill-conditioned, and the inverse \(\Sigma^{-1}\) used by MV optimization magnifies small estimation errors. The resulting portfolio can therefore be extremely unstable and economically implausible.Speaking of implementation practicality, it is also hard to monitor, constrain, and communicate thousands of security-level weights than a smaller set of economically meaningful asset-class allocations.


## 1.2 When does HMC's two-stage optimization work well?

HMC's two-stage procedure is:

1. construct an optimized portfolio within each asset class;
2. optimize across those asset-class portfolios and combine.

For this to approximate the full security-level mean-variance problem, the asset-class partition must preserve the important covariance structure. In particular, securities within an asset class should have sufficiently similar economic exposures that one optimized class portfolio captures most of the relevant within-class opportunities. In addition, different asset classes should represent genuinely different sources of risk and return. There should not be important cross-class hedges that would cause the optimal choice of securities within one class to change dramatically depending on the weights assigned to securities in another class.

In other words, the first-stage asset-class portfolios need to be good sufficient summaries of the underlying securities. If the partition puts economically different securities together, or separates securities whose relative weights should be chosen jointly, the two-stage solution can differ substantially from the full optimum.


## 1.3 Should TIPS be a separate asset class?

Conceptually, there is a strong case for treating TIPS separately if their real-return behavior is sufficiently different from nominal bonds and other existing assets. TIPS provide a direct hedge against realized inflation because their principal and coupon payments adjust with the price level. In HMC's capital-market assumptions, they also have low estimated volatility and a correlation structure that differs from both nominal bonds and risky assets, which can make them valuable even if their standalone expected return is not unusually high.

The main caution is parameter uncertainty. Since TIPS were still a new U.S. security at the time of the case, HMC had only a short domestic history from which to estimate expected return, volatility, and correlations. A large unconstrained MV weight could therefore reflect estimation error as much as true diversification value, which therefore should be treated cautiously. 


## 1.4 Why does HMC focus on real returns?

Harvard cares about real returns because the endowment must preserve its purchasing power while funding university spending. Nominal returns can overstate investment performance when inflation is high.

The relationship between nominal and real returns is approximately

$$
r_i^{real} \approx r_i^{nominal} - \pi.
$$

If inflation were constant, converting nominal returns to real returns would mostly shift expected returns. However, inflation varies over time and affects different assets differently. As a result, using real returns can also change estimated volatilities, correlations, and therefore the mean-variance allocation. This is particularly relevant for TIPS, whose payments adjust with inflation. Their risk and diversification properties can therefore look different from those of nominal Treasury bonds when HMC evaluates portfolios in real-return terms.

## 1.5 How should Harvard's allocation bounds enter the optimization?

Harvard's allocation limits can be incorporated directly as constraints in the mean-variance problem:

$$
\min_w \quad w^\top \Sigma w
$$
subject to
$$
w^\top \mu = \mu_{targ}, \quad
\mathbf{1}^\top w = 1,
$$

and asset-specific bounds

$$
\ell_i \leq w_i \leq u_i.
$$

In Exhibit 5, risky assets are constrained between 0% and 100%, while cash can range from -50% to 100%. In Exhibit 6, most existing asset classes are restricted to roughly $\pm$ 10 percentage points around their May 1999 policy weights, while TIPS can range from 0% to 100%. These constraints prevent extreme allocations and make the optimized portfolio more practical to implement, at the cost of some mean-variance efficiency.

## 1.6 Why are the zero-weight constraints on domestic equities and bonds binding?

Exhibit 5 imposes a no-short lower bound of zero on the risky asset classes. Domestic equities and domestic bonds sit at that lower bound across the full range of target returns, which means the unconstrained optimizer would prefer negative weights in those assets. The reason is not that either asset must have a negative expected return. MV optimization evaluates each asset conditional on what the rest of the portfolio can already provide. 

Under HMC's assumed means and covariance matrix, domestic equities are relatively unattractive compared with combinations of other growth assets such as PE and EM; nominal domestic bonds are relatively unattractive once TIPS provide low-volatility fixed-income exposure with direct inflation protection. Their marginal contribution to the optimized portfolio is therefore weak enough that, the optimizer would use them as funding/hedging shorts without constraints. With shorting prohibited, the optimum is pushed to exactly zero.

## 1.7 How costly are the tighter constraints in Exhibit 6?

The tighter, policy-oriented constraints in Exhibit 6 make the portfolio less MV efficient, but the compromise is fairly modest. For the same real-return targets, the standard deviation rises as follows:

| Target real return | Exhibit 5 vol. | Exhibit 6 vol. | Increase |
|---:|---:|---:|---:|
| 5.75% | 5.87% | 6.46% | +0.59 pp |
| 6.00% | 6.52% | 7.08% | +0.56 pp |
| 6.25% | 7.17% | 7.74% | +0.57 pp |
| 6.50% | 7.83% | 8.42% | +0.59 pp |
| 6.75% | 8.49% | 9.12% | +0.63 pp |
| 7.00% | 9.18% | 9.84% | +0.66 pp |
| 7.25% | 9.88% | 10.61% | +0.73 pp |

The increase is roughly 7–10% more volatility depending on the target. The Sharpe ratio falls only from about 0.38 in Exhibit 5 to roughly 0.35 in Exhibit 6. Thus, we see that economically sensible constraints can make the portfolio much more implementable at a relatively small cost in estimated efficiency.


# 2. Mean-Variance Optimization

The case gives us the economic argument for TIPS. We now ask whether a modern sample of liquid ETF returns tells the same story. The provided data represent the major asset classes with ETFs. We:

- treat `SHV` as the risk-free asset;
- use the supplied excess returns;
- drop `QAI` as instructed;
- annualize means by 12 and volatilities by $\sqrt{12}$.

The analysis below is deliberately kept in reusable functions so the notebook remains easy to audit and modify for the group.


## 2.0 Data and helper functions

In [2]:
# Packages needed and locate route to dataset
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
ANNUALIZATION = 12
TARGET_MONTHLY_MEAN = 0.01

# Prefer the repo path, with fallbacks for running the notebook elsewhere.
candidates = [
    Path("data/multi_asset_etf_data.xlsx"),
    Path("../data/multi_asset_etf_data.xlsx"),
    Path("multi_asset_etf_data.xlsx"),
]

DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find multi_asset_etf_data.xlsx. "
        "Place it in data/ relative to the repository root or update DATA_PATH."
    )

print(f"Using data file: {DATA_PATH.resolve()}")

Using data file: /Users/yuhanxiao/Downloads/multi_asset_etf_data.xlsx


In [4]:
excess = pd.read_excel(
    DATA_PATH,
    sheet_name="excess returns",
    index_col=0,
    parse_dates=True,
)

# Exclude QAI.
returns = excess.drop(columns="QAI").copy()
print(f"Sample: {returns.index.min():%Y-%m} to {returns.index.max():%Y-%m}")
print(f"Monthly observations: {len(returns)}")
print(f"Risky assets ({returns.shape[1]}): {', '.join(returns.columns)}")

display(returns.tail())

Sample: 2011-02 to 2026-07
Monthly observations: 186
Risky assets (10): BWX, DBC, EEM, EFA, HYG, IEF, IYR, PSP, SPY, TIP


,BWX,DBC,EEM,EFA,HYG,IEF,IYR,PSP,SPY,TIP
Date,,,,,,,,,,
2026-03-31,-0.0480,0.1506,-0.0953,-0.0810,-0.0123,-0.0260,-0.0665,-0.0641,-0.0521,-0.0163
2026-04-30,0.0137,0.0713,0.1238,0.0505,0.0123,-0.0044,0.0824,0.0749,0.1021,0.0079
2026-05-31,-0.0054,-0.0550,0.0692,0.0213,0.0015,-0.0031,-0.0146,-0.0019,0.0497,-0.0010
2026-06-30,-0.0228,-0.0984,-0.0003,0.0042,-0.0019,-0.0002,0.0103,-0.0715,-0.0130,-0.0073
2026-07-31,0.0005,0.1013,-0.0665,0.0130,-0.0049,-0.0172,0.0244,0.0677,-0.0030,-0.0102


In [5]:
def annualized_summary(rets):
    """Annualized mean, volatility, and Sharpe ratio from monthly excess returns."""
    mean_ann = rets.mean() * ANNUALIZATION
    vol_ann = rets.std(ddof=1) * np.sqrt(ANNUALIZATION)
    sharpe_ann = mean_ann / vol_ann

    return pd.DataFrame({
        "Annualized Mean": mean_ann,
        "Annualized Volatility": vol_ann,
        "Annualized Sharpe": sharpe_ann,
    })


def tangency_weights_from_moments(mu, cov):
    """Tangency weights normalized to sum to one."""
    direction = np.linalg.solve(cov, mu)
    return direction / direction.sum()


def tangency_weights(rets):
    """Tangency portfolio estimated from a return DataFrame."""
    return pd.Series(
        tangency_weights_from_moments(
            rets.mean().values,
            rets.cov().values,
        ),
        index=rets.columns,
        name="Tangency Weight",
    )


def portfolio_stats(weights, rets, mean_override=None):
    """Annualized mean, volatility, and Sharpe ratio for a fixed weight vector."""
    w = np.asarray(weights, dtype=float)
    mu = (
        rets.mean().values
        if mean_override is None
        else np.asarray(mean_override, dtype=float)
    )
    cov = rets.cov().values

    mean_monthly = w @ mu
    vol_monthly = np.sqrt(w @ cov @ w)

    mean_ann = mean_monthly * ANNUALIZATION
    vol_ann = vol_monthly * np.sqrt(ANNUALIZATION)

    return pd.Series({
        "Annualized Mean": mean_ann,
        "Annualized Volatility": vol_ann,
        "Annualized Sharpe": mean_ann / vol_ann,
    })


def scale_to_target_mean(weights, mu, target=TARGET_MONTHLY_MEAN):
    """Scale the entire risky weight vector to hit a target monthly excess return."""
    w = np.asarray(weights, dtype=float)
    mu = np.asarray(mu, dtype=float)
    return w * target / (w @ mu)

## 2.1 Summary Statistics

In [6]:
summary = annualized_summary(returns)

display(summary.style.format({
    "Annualized Mean": "{:.2%}",
    "Annualized Volatility": "{:.2%}",
    "Annualized Sharpe": "{:.3f}",
}))

best_sharpe = summary["Annualized Sharpe"].idxmax()
worst_sharpe = summary["Annualized Sharpe"].idxmin()

print(
    f"Highest Sharpe ratio: {best_sharpe} "
    f"({summary.loc[best_sharpe, 'Annualized Sharpe']:.3f})"
)
print(
    f"Lowest Sharpe ratio: {worst_sharpe} "
    f"({summary.loc[worst_sharpe, 'Annualized Sharpe']:.3f})"
)

,Annualized Mean,Annualized Volatility,Annualized Sharpe
BWX,-1.72%,8.19%,-0.210
DBC,1.30%,17.23%,0.075
EEM,4.44%,17.91%,0.248
EFA,6.38%,14.86%,0.429
HYG,3.45%,7.33%,0.471
IEF,0.86%,6.21%,0.138
IYR,7.01%,16.55%,0.423
PSP,7.64%,21.13%,0.362
SPY,12.70%,14.14%,0.898
TIP,1.28%,4.97%,0.257


Highest Sharpe ratio: SPY (0.898)
Lowest Sharpe ratio: BWX (-0.210)


**Takeaway.** SPY has the strongest standalone risk-adjusted performance in the sample, while BWX has the weakest. But Sharpe ratios alone are only a first pass, as mean-variance allocation depends just as importantly on how the assets move with one another.


## 2.2 Diversification and the bond comparison

The next step is to look at correlations, then compare TIPS directly with domestic and foreign nominal bonds.


In [7]:
corr = returns.corr()
display(corr.style.format("{:.3f}"))

# Unique off-diagonal correlation pairs
pair_corrs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .sort_values()
)

lowest_pair = pair_corrs.index[0]
highest_pair = pair_corrs.index[-1]

print(
    f"Lowest correlation: {lowest_pair[0]} / {lowest_pair[1]} "
    f"= {pair_corrs.iloc[0]:.3f}"
)
print(
    f"Highest correlation: {highest_pair[0]} / {highest_pair[1]} "
    f"= {pair_corrs.iloc[-1]:.3f}"
)

,BWX,DBC,EEM,EFA,HYG,IEF,IYR,PSP,SPY,TIP
BWX,1.000,0.161,0.612,0.613,0.600,0.581,0.555,0.524,0.444,0.675
DBC,0.161,1.000,0.439,0.440,0.419,-0.309,0.253,0.420,0.384,0.084
EEM,0.612,0.439,1.000,0.812,0.670,0.051,0.574,0.697,0.693,0.384
EFA,0.613,0.440,0.812,1.000,0.776,0.063,0.700,0.862,0.833,0.402
HYG,0.600,0.419,0.670,0.776,1.000,0.193,0.731,0.798,0.783,0.540
IEF,0.581,-0.309,0.051,0.063,0.193,1.000,0.322,0.012,0.011,0.756
IYR,0.555,0.253,0.574,0.700,0.731,0.322,1.000,0.726,0.743,0.600
PSP,0.524,0.420,0.697,0.862,0.798,0.012,0.726,1.000,0.872,0.397
SPY,0.444,0.384,0.693,0.833,0.783,0.011,0.743,0.872,1.000,0.386
TIP,0.675,0.084,0.384,0.402,0.540,0.756,0.600,0.397,0.386,1.000


Lowest correlation: DBC / IEF = -0.309
Highest correlation: PSP / SPY = 0.872


In [8]:
bond_comparison = summary.loc[["TIP", "IEF", "BWX"]]

display(bond_comparison.style.format({
    "Annualized Mean": "{:.2%}",
    "Annualized Volatility": "{:.2%}",
    "Annualized Sharpe": "{:.3f}",
}))

,Annualized Mean,Annualized Volatility,Annualized Sharpe
TIP,1.28%,4.97%,0.257
IEF,0.86%,6.21%,0.138
BWX,-1.72%,8.19%,-0.210


**Takeaway.** PSP and SPY have the highest pairwise correlation, which is consistent with PSP's strong equity-like exposure. DBC and IEF have the lowest correlation, illustrating how commodities can diversify nominal government bonds.

Within fixed income, TIP looks comparatively strong in this sample - it has a higher annualized excess return than both IEF and BWX, lower volatility than both, and consequently the highest Sharpe ratio of the three. This gives TIPS a favorable profile, but whether they still add value once the entire covariance matrix is considered is yet to be investigated.


## 2.3 The tangency portfolio

We now move from individual assets to the portfolio problem. With excess returns $\mu$ and covariance matrix $\Sigma$, the tangency direction is

$$
w_{\text{tan}} \propto \Sigma^{-1}\mu.
$$

We normalize the risky-asset weights to sum to one.


In [11]:
w_tan = tangency_weights(returns)
tan_stats = portfolio_stats(w_tan.values, returns)

display(
    w_tan.to_frame()
         .style.format({"Tangency Weight": "{:.2%}"})
)

tan_stats_display = pd.DataFrame({
    "Tangency Portfolio": [
        f"{tan_stats['Annualized Mean']:.2%}",
        f"{tan_stats['Annualized Volatility']:.2%}",
        f"{tan_stats['Annualized Sharpe']:.4f}",
    ]
}, index=["Annualized Mean", "Annualized Volatility", "Annualized Sharpe"])

display(tan_stats_display)

,Tangency Weight
BWX,-117.97%
DBC,3.63%
EEM,-0.10%
EFA,18.48%
HYG,20.38%
IEF,131.88%
IYR,-22.37%
PSP,-41.06%
SPY,126.32%
TIP,-19.20%


,Tangency Portfolio
Annualized Mean,16.17%
Annualized Volatility,10.96%
Annualized Sharpe,1.4758


In [12]:
ranking = pd.DataFrame({
    "Tangency Weight": w_tan,
    "Individual Sharpe": summary["Annualized Sharpe"],
})

ranking["Weight Rank"] = ranking["Tangency Weight"].rank(ascending=False)
ranking["Sharpe Rank"] = ranking["Individual Sharpe"].rank(ascending=False)

display(
    ranking.sort_values("Tangency Weight", ascending=False)
           .style.format({
               "Tangency Weight": "{:.2%}",
               "Individual Sharpe": "{:.3f}",
               "Weight Rank": "{:.0f}",
               "Sharpe Rank": "{:.0f}",
           })
)

,Tangency Weight,Individual Sharpe,Weight Rank,Sharpe Rank
IEF,131.88%,0.138,1,8
SPY,126.32%,0.898,2,1
HYG,20.38%,0.471,3,2
EFA,18.48%,0.429,4,3
DBC,3.63%,0.075,5,9
EEM,-0.10%,0.248,6,7
TIP,-19.20%,0.257,7,6
IYR,-22.37%,0.423,8,4
PSP,-41.06%,0.362,9,5
BWX,-117.97%,-0.210,10,10


**Takeaway.** Note that the tangency-weight ranking does NOT match the ranking of standalone Sharpe ratios. This is expected, since $\Sigma^{-1}\mu$ rewards not only a high expected return relative to own volatility, but also a useful covariance pattern with the rest of the portfolio. 

To illustrate, IEF is a good example. Its individual Sharpe ratio is not especially high, yet its diversification role supports a large positive tangency weight. Conversely, several assets with positive individual Sharpes receive negative weights because their exposures are redundant relative to more attractive combinations of other assets.

## 2.4 Do TIPS actually expand the opportunity set?

We examine two counterfactuals:

1. **Remove TIP entirely.** If the maximum Sharpe ratio barely changes, TIP is not adding much to the opportunity set at the historical sample estimates.
2. **Raise TIP's expected monthly excess return by 0.0012**, holding the covariance matrix fixed. This checks how sensitive the conclusion is to the expected-return estimate.


In [13]:
# Baseline
baseline_stats = portfolio_stats(w_tan.values, returns)

# Counterfactual 1: remove TIP
returns_no_tip = returns.drop(columns="TIP")
w_no_tip = tangency_weights(returns_no_tip)
no_tip_stats = portfolio_stats(w_no_tip.values, returns_no_tip)

# Counterfactual 2: increase TIP expected monthly excess return by 0.0012
mu_base = returns.mean()
mu_tip_up = mu_base.copy()
mu_tip_up.loc["TIP"] += 0.0012

w_tip_up = pd.Series(
    tangency_weights_from_moments(
        mu_tip_up.values,
        returns.cov().values,
    ),
    index=returns.columns,
    name="TIP mean up 0.0012",
)

tip_up_stats = portfolio_stats(
    w_tip_up.values,
    returns,
    mean_override=mu_tip_up.values,
)

tips_comparison = pd.DataFrame({
    "Baseline": baseline_stats,
    "No TIP": no_tip_stats,
    "TIP mean up 0.0012": tip_up_stats,
}).T

display(
    tips_comparison.style.format({
        "Annualized Mean": "{:.2%}",
        "Annualized Volatility": "{:.2%}",
        "Annualized Sharpe": "{:.4f}",
    })
)

,Annualized Mean,Annualized Volatility,Annualized Sharpe
Baseline,16.17%,10.96%,1.4758
No TIP,15.54%,10.54%,1.4745
TIP mean up 0.0012,13.27%,8.42%,1.5749


In [15]:
weights_comparison = pd.concat(
    [
        w_tan.rename("Baseline"),
        w_no_tip.reindex(returns.columns).rename("No TIP"),
        w_tip_up.rename("TIP mean up 0.0012"),
    ],
    axis=1,
)

display(weights_comparison.style.format("{:.2%}"))

,Baseline,No TIP,TIP mean up 0.0012
BWX,-117.97%,-113.75%,-86.02%
DBC,3.63%,2.58%,-4.33%
EEM,-0.10%,-0.62%,-4.02%
EFA,18.48%,18.81%,20.94%
HYG,20.38%,17.91%,1.72%
IEF,131.88%,116.16%,12.89%
IYR,-22.37%,-22.34%,-22.16%
PSP,-41.06%,-40.04%,-33.38%
SPY,126.32%,121.28%,88.19%
TIP,-19.20%,nan%,126.17%


**Takeaway.** At the historical estimates, removing TIP barely changes the maximum Sharpe ratio. On that evidence alone, TIPS only marginally expand the opportunity set. However, the conclusion changes sharply when TIP's expected monthly excess return is increased by 12 bps. The optimizer moves heavily toward TIPS and the attainable Sharpe ratio rises materially.

This sensitivity is exactly the concern raised in the case. TIPS can be a genuinely distinct and useful asset class, but MV recommendations depend heavily on an expected return that is difficult to estimate precisely. A reasonable institutional response is therefore not to ignore TIPS, but to recognize them as a distinct exposure while placing bounds on the allocation, and proceeding cautiously based on Harvard's forward-looking expected return for TIPS.


# 3. Allocation Methods at a Common Return Target

The final comparison asks a slightly different question. Rather than maximize Sharpe ratio directly, suppose the investor wants a monthly expected excess return of

$$
\mu_{\text{targ}}=0.01.
$$

We compare three rules:

- **Equal weight (EW):** $w_i=1/n$
- **"Risk parity" (RP):** $w_i\propto 1/\sigma_i^2$, using the assignment's inverse-variance definition
- **Mean-variance (MV):** the tangency direction from Section 2.3

To make the methods comparable, each risky weight vector is rescaled to hit the same target expected excess return.


In [16]:
mu = returns.mean().values
cov = returns.cov().values
n_assets = returns.shape[1]

# Base allocation rules
w_ew_base = np.ones(n_assets) / n_assets

inverse_variance = 1 / np.diag(cov)
w_rp_base = inverse_variance / inverse_variance.sum()

w_mv_base = w_tan.values

# Scale each risky portfolio to the same target monthly expected excess return.
w_ew = scale_to_target_mean(w_ew_base, mu)
w_rp = scale_to_target_mean(w_rp_base, mu)
w_mv = scale_to_target_mean(w_mv_base, mu)

allocation_weights = pd.DataFrame(
    {"EW": w_ew, "RP": w_rp, "MV": w_mv},
    index=returns.columns,
)

display(allocation_weights.style.format("{:.2%}"))

,EW,RP,MV
BWX,27.69%,67.33%,-87.56%
DBC,27.69%,15.21%,2.69%
EEM,27.69%,14.08%,-0.07%
EFA,27.69%,20.45%,13.72%
HYG,27.69%,83.97%,15.13%
IEF,27.69%,117.00%,97.88%
IYR,27.69%,16.48%,-16.60%
PSP,27.69%,10.11%,-30.47%
SPY,27.69%,22.58%,93.75%
TIP,27.69%,182.44%,-14.25%


In [17]:
target_check = allocation_weights.T @ returns.mean()

exposure_summary = pd.DataFrame({
    "Total Risky Exposure": allocation_weights.sum(),
    "Monthly Expected Excess Return": target_check,
})

display(
    exposure_summary.style.format({
        "Total Risky Exposure": "{:.3f}",
        "Monthly Expected Excess Return": "{:.2%}",
    })
)

,Total Risky Exposure,Monthly Expected Excess Return
EW,2.769,1.00%
RP,5.497,1.00%
MV,0.742,1.00%


In [18]:
allocation_performance = pd.DataFrame({
    "EW": portfolio_stats(w_ew, returns),
    "RP": portfolio_stats(w_rp, returns),
    "MV": portfolio_stats(w_mv, returns),
}).T

display(
    allocation_performance.style.format({
        "Annualized Mean": "{:.2%}",
        "Annualized Volatility": "{:.2%}",
        "Annualized Sharpe": "{:.3f}",
    })
)

,Annualized Mean,Annualized Volatility,Annualized Sharpe
EW,12.00%,27.53%,0.436
RP,12.00%,32.61%,0.368
MV,12.00%,8.13%,1.476


**Takeaway.** 
Because all three portfolios are scaled to the same 1% monthly excess return, the comparison comes down to how much risk each method needs to get there.

**EW** is the simplest benchmark: it ignores both expected returns and risk differences, assigning the same initial weight to every asset. **RP** is more deliberate, tilting toward lower-volatility assets through inverse-variance weights. But low volatility does not necessarily come with high expected return, so RP has to be scaled up aggressively to hit the target. This leaves it with even higher volatility than EW in our sample.

**MV** is different because it utilizes the full portfolio structure: expected returns, individual risks, and, importantly, covariances across assets. That allows it to use diversification rather than simply favoring assets that look safe on their own. As a result, MV reaches the same 12% annualized excess return with only 8.13% volatility, versus 27.53% for EW and 32.63% for RP.

The key distinction is therefore between selecting low-volatility assets and building a low-volatility portfolio. In this sample, MV does the latter far more efficiently.

# Overall Conclusion

The case and our empirical results point to the same broader idea: portfolio construction is not just about finding the assets with the highest returns, but about balancing return, diversification, and implementation.

For HMC, this means thinking in real returns, organizing the portfolio around economically distinct asset classes, and recognizing that optimization is only as reliable as the assumptions behind it. TIPS fit naturally into that framework because they provide a distinct inflation-linked exposure, but our sensitivity analysis also reveals how strongly their optimal weight depends on expected-return estimates.

The quantitative results reinforce the same point. Mean-variance optimization is very efficient in-sample because it uses the full covariance structure, but it can also produce extreme and unstable allocations. That is why constraints provide a healthy balance: they sacrifice some theoretical efficiency in exchange for a portfolio that is more robust, interpretable, and practical to implement.

Overall, the optimizer is best viewed as a guide to the tradeoffs rather than a portfolio to follow mechanically.